In [75]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# 1. 데이터 가져오기

In [76]:
import pandas as pd
df = pd.read_csv(r"C:\ai_x\download\sharedata\부동산_250213\최종전국평당분양가격_결측치제외.csv", encoding='cp949')
df.head()

,지역명,연도,월,평당분양가격
0,서울,2013,12,18189.0
1,부산,2013,12,8111.0
2,대구,2013,12,8080.0
3,인천,2013,12,10204.0
4,광주,2013,12,6098.0


# 2. 데이터 전처리_ 지역명의 라벨 인코딩
- 지역명을 라벨인코딩한 지역명2 추가
- 분석할 경우 원핫인코딩까지 할 것을 추천

In [77]:
from sklearn.preprocessing import LabelEncoder
import numpy as np
field_scaler = LabelEncoder()
df_output = df.iloc[:,-1]
df_input = df.iloc[:,:-1]

In [78]:
# 입력변수
field_name = np.array(df['지역명'])
field_name_label = field_scaler.fit_transform(field_name)
df_input_label = pd.concat([df_input,pd.DataFrame(field_name_label)],axis=1)
df_input_label.columns = ['지역명', '연도','월','지역명2']
df_input_label
np_input_label = np.array(df_input_label.iloc[:,1:])
np_input_label

array([[2013,   12,    8],
       [2013,   12,    7],
       [2013,   12,    5],
       ...,
       [2024,    8,    3],
       [2024,    8,    2],
       [2024,    8,   14]], dtype=int64)

In [79]:
# 타겟변수
np_output = np.array(df_output).reshape(-1,1)
np_output

array([[18189. ],
       [ 8111. ],
       [ 8080. ],
       ...,
       [13827. ],
       [13252.8],
       [25419.9]])

In [94]:
# 원핫인코딩
from tensorflow.keras.utils import to_categorical
onehot_field_label = to_categorical(np_input_label[:,2])
onehot_field_label
df_final = pd.concat([df_semi_final_input,pd.DataFrame(onehot_field_label)],axis=1)
# display(df_final)
# display(pd_scaled_y),display(df_normalization_output)
df_final_output = pd.concat([df_output,pd_scaled_y,df_normalization_output],axis=1)
df_final_output.columns = ['평당분양가격','평당분양가격s','평당분양가격n']
display(df_final_output)

,평당분양가격,평당분양가격s,평당분양가격n
0,18189.0,1.168591,0.328198
1,8111.0,-0.728312,0.065274
2,8080.0,-0.734147,0.064466
3,10204.0,-0.334363,0.119878
4,6098.0,-1.107203,0.012757
...,...,...,...
2171,12058.2,0.014639,0.168252
2172,13120.8,0.214643,0.195974
2173,13827.0,0.347566,0.214398
2174,13252.8,0.239489,0.199418


# 3. normalization 스케일 조정
- 입력변수(지역명2, 연도, 월)와 타겟변수(평당분양가격) 따로 스케일 조정(MinMaxScaler)
- 지역명2n, 연도n, 월n 필드 추가

In [81]:
from sklearn.preprocessing import MinMaxScaler
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()
scaled_x = x_scaler.fit_transform(np_input_label)
scaled_y = y_scaler.fit_transform(np_output)
# pd.concat(df_input_label,)
pd_scaled_x = pd.DataFrame(scaled_x, columns = ['연도n','월n','지역명2n'])
df_normalization_input = pd.concat([df_input_label,pd_scaled_x], axis=1)
df_normalization_output = pd.DataFrame(scaled_y,columns=['평당분양가격'])
df_normalization_output

,평당분양가격
0,0.328198
1,0.065274
2,0.064466
3,0.119878
4,0.012757
...,...
2171,0.168252
2172,0.195974
2173,0.214398
2174,0.199418


# 4. standardization 스케일 조정
- 입력변수(지역명2, 연도, 월)와 타겟변수(평당분양가격) 따로 스케일 조정(StandardScaler)
- 지역명2s, 연도S, 월s 필드 추가

In [82]:
from sklearn.preprocessing import StandardScaler
x_scaler = StandardScaler()
y_scaler = StandardScaler()
scaled_x = x_scaler.fit_transform(np_input_label)
scaled_y = y_scaler.fit_transform(np_output)
pd_scaled_x = pd.DataFrame(scaled_x,columns=['연도s','월s','지역명2s'])
pd_scaled_y = pd.DataFrame(scaled_y,columns=['평당분양가격'])
df_semi_final_input = pd.concat([df_normalization_input, pd_scaled_x],axis=1)
display(df_semi_final_input), display(pd_scaled_y)

,지역명,연도,월,지역명2,연도n,월n,지역명2n,연도s,월s,지역명2s
0,서울,2013,12,8,0.0,1.000000,0.5000,-1.875367,1.62196,0.000000
1,부산,2013,12,7,0.0,1.000000,0.4375,-1.875367,1.62196,-0.204124
2,대구,2013,12,5,0.0,1.000000,0.3125,-1.875367,1.62196,-0.612372
3,인천,2013,12,11,0.0,1.000000,0.6875,-1.875367,1.62196,0.612372
4,광주,2013,12,4,0.0,1.000000,0.2500,-1.875367,1.62196,-0.816497
...,...,...,...,...,...,...,...,...,...,...
2171,전북,2024,8,13,1.0,0.636364,0.8125,1.664199,0.46374,1.020621
2172,전남,2024,8,12,1.0,0.636364,0.7500,1.664199,0.46374,0.816497
2173,경북,2024,8,3,1.0,0.636364,0.1875,1.664199,0.46374,-1.020621
2174,경남,2024,8,2,1.0,0.636364,0.1250,1.664199,0.46374,-1.224745


,평당분양가격
0,1.168591
1,-0.728312
2,-0.734147
3,-0.334363
4,-1.107203
...,...
2171,0.014639
2172,0.214643
2173,0.347566
2174,0.239489


(None, None)